## 使用langchain + ollama + fastapi翻译接口

In [1]:
import os

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from fastapi import FastAPI
from langserve import add_routes
import uvicorn

In [2]:
# 定义模板
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "请将以下内容翻译为{language}"),
        ("user", "{text}")
    ]
)

# 定义模型
model_name = 'qwen2.5:1.5b'
ollama_base_url = os.environ.get("OLLAMA_API_BASE")

model = init_chat_model(model=model_name, model_provider="ollama", base_url=ollama_base_url)

# 定义parser
str_parser = StrOutputParser()

# 定义链
chain = prompt_template | model | str_parser

In [3]:
# chain调用
chain.invoke({"language": "中文", "text": "This is a good book"})

'这是本好书'

In [4]:
app = FastAPI()


@app.get("/")
def index():
    return {"message": "Hello, World!"}
    

@app.post("/translate")
def translate(text: str, language: str = "中文"):
    if text:
        data = {"language": language, "text": text}
        result = chain.invoke(data)
        return result
    else:
        msg = "请传递text内容"
        return msg


# 利用langserve
add_routes(
    app,
    chain,
    path="/chain"
)

In [5]:
# uvicorn.run(app, host="0.0.0.0", port=8000)

In [6]:
# 在jupyter中运行fastapi
# import asyncio

# config = uvicorn.Config(app)
# server = uvicorn.Server(config)
# loop = asyncio.get_running_loop()
# loop.create_task(server.serve())

In [7]:
!curl http://127.0.0.1:8000

{"message":"Hello, World!"}

In [8]:
!curl --request POST 'http://127.0.0.1:8000/translate?text=this%20is%20a%20good%20book'

"这是本好书。"

In [9]:
!curl --location 'http://127.0.0.1:8000/chain/invoke' \
--header 'Content-Type: application/json' \
--data '{"input": {"language": "中文","text": "This is a book"}}'

{"output":"这是本书","metadata":{"run_id":"b353956e-d258-4218-8ec4-e2f243af0121","feedback_tokens":[]}}

In [10]:
!curl --location 'http://127.0.0.1:8000/chain/invoke' \
--header 'Content-Type: application/json' \
--data '{"input": {"language": "英文","text": "我喜欢编程，编程是一件很快乐的事情"}, "config": {}}'

{"output":"I like programming, and programming is a very pleasant thing to do.","metadata":{"run_id":"ee790d3b-cb38-4ffe-8d58-d9388a74cf5d","feedback_tokens":[]}}